In [1]:
import torch
import torch.nn as nn
import json
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [ ]:
with open('champ_data.json', 'r') as f:
    champ_data = json.load(f)

with open('champ_names.json', 'r') as f:
    champ_names = json.load(f)

In [69]:
NUM_CHAMPIONS_PER_GAME = 10
NUM_ROLES = 5
EMBEDDING_DIM = 16
HIDDEN_DIM = 128
TOTAL_CHAMPIONS = len(champ_names) + 1 
BATCH_SIZE = 8

In [17]:
str_to_idx = dict(zip(champ_names, range(len(champ_names))))
str_to_idx['Masked'] = len(champ_names)  # Add a special token for masked slots

In [48]:
encode = lambda name: str_to_idx[name] # takes a champion name and returns the corresponding index
decode = lambda idx: champ_names[idx] # takes index and returns the corresponding champion name

print(decode(encode('Ahri')))

Ahri


In [29]:
class ChampionDataset(Dataset):
    def __init__(self, data, encode_fn):
        self.data = data
        self.encode = encode_fn

    def __len__(self):
        return len(self.data) * NUM_CHAMPIONS_PER_GAME

    def __getitem__(self, idx):
        match_idx = idx // NUM_CHAMPIONS_PER_GAME
        champ_idx = idx % NUM_CHAMPIONS_PER_GAME
        current_data = self.data[match_idx]
        masked_data = current_data.copy()
        masked_data[champ_idx] = 'Masked' # Mask the current champion
        x = [self.encode(name) for name in masked_data]
        return torch.tensor(x), torch.tensor(self.encode(current_data[champ_idx]))

In [70]:
champion_dataset = ChampionDataset(champ_data, encode)
dataloader = DataLoader(champion_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class LeagueDraftModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(num_embeddings=TOTAL_CHAMPIONS, embedding_dim=EMBEDDING_DIM)
        self.role_embedding = nn.Embedding(num_embeddings=NUM_ROLES, embedding_dim=EMBEDDING_DIM)
        self.lm = nn.Linear(EMBEDDING_DIM * NUM_CHAMPIONS_PER_GAME, HIDDEN_DIM)
        self.output_layer = nn.Linear(HIDDEN_DIM, TOTAL_CHAMPIONS)

    def forward(self, x, labels):
        # x is of shape (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME), labels is of shape (BATCH_SIZE)
        batch_size = x.size(0)
        token_embeds = self.token_embedding(x)  # Shape: (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME, EMBEDDING_DIM)
        role_embeds = self.role_embedding(torch.arange(start=0, end=NUM_ROLES).repeat(2)) # Shape: (NUM_CHAMPIONS_PER_GAME, EMBEDDING_DIM)
        embeds = token_embeds + role_embeds # Shape: (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME, EMBEDDING_DIM)

        # flatten
        embeds = embeds.view(batch_size, NUM_CHAMPIONS_PER_GAME * EMBEDDING_DIM) # Shape: (BATCH_SIZE, NUM_CHAMPIONS_PER_GAME * EMBEDDING_DIM)

        pre_activations = torch.tanh(self.lm(embeds))  # Shape: (BATCH_SIZE, HIDDEN_DIM)
        logits = self.output_layer(pre_activations) # Shape: (BATCH_SIZE, TOTAL_CHAMPIONS)
        loss = F.cross_entropy(logits, labels)
        return loss

In [76]:
x_batch, y_batch = next(iter(dataloader))
print(y_batch.shape, x_batch.shape)

torch.Size([8]) torch.Size([8, 10])


In [93]:
torch.manual_seed(1)
x = torch.randn(2, 2, 2)
print(x.shape)
print(x)
print(x.view(2, 4))

torch.Size([2, 2, 2])
tensor([[[ 0.6614,  0.2669],
         [ 0.0617,  0.6213]],

        [[-0.4519, -0.1661],
         [-1.5228,  0.3817]]])
tensor([[ 0.6614,  0.2669,  0.0617,  0.6213],
        [-0.4519, -0.1661, -1.5228,  0.3817]])
